# Grafo com loop e decisão condicional Utilizando [LangGraph](https://docs.langchain.com/oss/python/langgraph)



### 1. Instalando as dependências

In [ ]:
!pip install -U langchain-huggingface

In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

### 2. Configurar a Hugging Face

In [6]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [7]:
llm_hf = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    max_new_tokens=512,
    temperature=0
)

llm = ChatHuggingFace(llm=llm_hf)

In [8]:
resposta = llm.invoke(
    "Explique o que é um agente inteligente em duas frases."
)

print(resposta.content)

Um agente inteligente é um sistema que percebe seu ambiente por meio de sensores, processa essas informações e toma decisões ou executa ações usando atuadores para alcançar objetivos definidos. Ele combina percepção, raciocínio e aprendizado para adaptar seu comportamento e otimizar o desempenho em situações incertas ou dinâmicas.


### 3. Estado do nosso LangGraph

Nosso agente terá um estado chamado messages.
O add_messages permite que o LangGraph vá acumulando as mensagens durante a execução.


In [9]:
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages


class StateMessage(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### 4. Criar as ferramentas

In [10]:
from langchain_core.tools import tool


@tool
def somar(a: float, b: float) -> float:
    """Útil para somar dois números."""
    return a + b


@tool
def subtrair(a: float, b: float) -> float:
    """Útil para subtrair dois números."""
    return a - b


@tool
def multiplicar(a: float, b: float) -> float:
    """Útil para multiplicar dois números."""
    return a * b


@tool
def dividir(a: float, b: float) -> str:
    """Útil para dividir dois números."""

    if b == 0:
        return "Erro: divisão por zero não é permitida."

    return str(a / b)

In [11]:
tools = [
    somar,
    subtrair,
    multiplicar,
    dividir
]

### 5. bind_tools

Aqui é onde precisamos verificar se nosso modelo suporta chamada de ferramentas.

In [12]:
model = llm.bind_tools(tools)

In [28]:
teste = model.invoke(
    "Quanto é 30 multiplicado por 3?"
)

print(teste)

content='30 multiplicado por 3 é **90**.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 216, 'total_tokens': 266}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b546658c8e93d2e57ef2', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0d5f1-4c68-7832-b4fd-adc192fc8407-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 216, 'output_tokens': 50, 'total_tokens': 266}


In [29]:
tool_calls=[
    {
        "name": "multiplicar",
        "args": {
            "a": 15,
            "b": 3
        }
    }
]

### 6. ToolNode

O ToolNode é o componente que realmente executa as ferramentas solicitadas pelo modelo.

In [30]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

### 7. Criar o nó do agente

In [31]:
def chamar_modelo(state: StateMessage):
    """Envia as mensagens para o modelo."""

    response = model.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

### 8. Decidir o próximo passo

In [32]:
def decidir_proximo_passo(
    state: StateMessage
) -> Literal["tools", "__end__"]:

    ultima_mensagem = state["messages"][-1]

    if ultima_mensagem.tool_calls:
        return "tools"

    return "__end__"

### 9. Construir o LangGraph

In [33]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(StateMessage)

Adicionamos os dois nós:

In [34]:
workflow.add_node(
    "agent",
    chamar_modelo
)

workflow.add_node(
    "tools",
    tool_node
)

Fluxo inicial:

In [35]:
workflow.add_edge(
    START,
    "agent"
)

Depois colocamos nossa decisão:

In [36]:
workflow.add_conditional_edges(
    "agent",
    decidir_proximo_passo
)

In [37]:
workflow.add_edge(
    "tools",
    "agent"
)

In [38]:
app = workflow.compile()

### 10. Agora vamos fazer o teste

In [39]:
pergunta = "Quanto é 15 multiplicado por 3?"

In [40]:
input_estado = {
    "messages": [
        ("user", pergunta)
    ]
}

E executamos:

In [41]:
resultado = app.invoke(input_estado)

print(resultado["messages"][-1].content)

15 × 3 = 45.


In [42]:
model = llm.bind_tools(tools)

teste = model.invoke(
    "Quanto é 15 multiplicado por 3?"
)

print(teste)

content='15\u202f×\u202f3 = **45**.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 216, 'total_tokens': 277}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b546658c8e93d2e57ef2', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0d5f1-bd0a-78f3-a62f-20d402e877f0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 216, 'output_tokens': 61, 'total_tokens': 277}


# Grafo RAG linear - LangGraph + PDF + embeddings + banco vetorial + retriever + LLM


In [1]:
from google.colab import files

arquivos = files.upload()

Saving O-Perigo-de-Estar-Lúcida-Rosa-Montero.pdf to O-Perigo-de-Estar-Lúcida-Rosa-Montero (1).pdf


In [2]:
nome_pdf = list(arquivos.keys())[0]

print(nome_pdf)

O-Perigo-de-Estar-Lúcida-Rosa-Montero (1).pdf


### ETAPA 2 — Instalar as bibliotecas

In [ ]:
!pip install -U pypdf langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers langgraph

### ETAPA 3 — Ler o livro

In [6]:
from pypdf import PdfReader

leitor = PdfReader(nome_pdf)

print(f"Quantidade de páginas: {len(leitor.pages)}")

Quantidade de páginas: 263


In [7]:
texto_paginas = []

for numero_pagina, pagina in enumerate(leitor.pages):
    texto = pagina.extract_text()

    if texto:
        texto_paginas.append({
            "pagina": numero_pagina + 1,
            "texto": texto
        })

print(f"Páginas com texto: {len(texto_paginas)}")

Páginas com texto: 258


### ETAPA 4 — Transformar o livro em chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [9]:
print(divisor)

In [10]:
from langchain_core.documents import Document

documentos = []

for pagina in texto_paginas:

    documento = Document(
        page_content=pagina["texto"],
        metadata={
            "pagina": pagina["pagina"],
            "arquivo": nome_pdf
        }
    )

    documentos.append(documento)

In [11]:
chunks = divisor.split_documents(documentos)

print(f"Quantidade de chunks: {len(chunks)}")

Quantidade de chunks: 643


### ETAPA 5 — Embeddings

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### ETAPA 6 — Criar nosso banco vetorial

In [13]:
from langchain_community.vectorstores import FAISS

banco_vetorial = FAISS.from_documents(
    chunks,
    embeddings
)

/tmp/ipykernel_7617/3131525589.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### ETAPA 7 — Testar o Retrieval ANTES do LangGraph

In [14]:
retriever = banco_vetorial.as_retriever(
    search_kwargs={
        "k": 3
    }
)

In [16]:
pergunta = "Qual é o principal conceito apresentado no livro?"

In [17]:
documentos_encontrados = retriever.invoke(pergunta)

In [18]:
for documento in documentos_encontrados:
    print("=" * 80)
    print(f"PÁGINA: {documento.metadata['pagina']}")
    print(documento.page_content[:1000])

PÁGINA: 51
me mandou o livro.  
Uma semana depois, recebi em casa aquele livro, reenviado pelo
cardiologista. Era Crónica del desamor, um exemplar da nona edição da
editora Debate. A dedicatória dizia: “Para o dr. Zarco, com minha
admiração, este meu primeiro livro ainda titubeante. Um beijo, Rosa
Montero”. A letra não tinha nada a ver com a minha, era pequena,
apertada e inclinada, mas o texto bem poderia ter sido escrito por
PÁGINA: 252
porque, em primeiro lugar, botei meus medos para fora através da
literatura, isto é, eu escrevi meu medo da loucura. E, em segundo,
acredito que tenho muitos pontos de contato com pessoas que estão
loucas, mas acho que posso… É algo por si só interessante, acho que
posso… não gosto da palavra sublimar, mas, enﬁm, acho que posso
simplesmente passar minha loucura para… talvez para outras pessoas.
Posso rebatê-la para fora de mim.
— Numa passagem do livro, a senhora conta que por muitos anos
chorou tão desconsolada pela morte dos gatos, que tinha
obrigat

### Teste 2 — uma pergunta específica

In [19]:
pergunta = "O que a autora fala sobre a influência da genética e do meio na formação do ser humano?"

In [20]:
documentos_encontrados = retriever.invoke(pergunta)

for documento in documentos_encontrados:
    print("=" * 80)
    print(f"PÁGINA: {documento.metadata['pagina']}")
    print(documento.page_content)

PÁGINA: 20
primeiros anos de vida daniﬁcam a estrutura do cérebro”, continua
Kandel. “De modo similar, precisamos de interação social para
continuarmos sendo inteligentes na velhice.” E o neurocientista David
Eagleman conta, no livro Incógnito, que os especialistas estão há
décadas procurando o gene relacionado à esquizofrenia e, com efeito,
já descobriram uma porção deles. Mas vários estudos mostram que
nenhum desses genes te predispõe a sofrer de determinada doença
tanto quanto a cor do seu passaporte. “A tensão social de ser
PÁGINA: 19
biologia e genética. Não tenho a menor dúvida de que, com efeito, a
inﬂuência ﬁsiológica é imensa, como prova a história da maldita
espasmoﬁlia: sem dúvida certos desequilíbrios hormonais, químicos,
sinápticos, produzem uma série de sintomas claramente
diagnosticáveis num bebê que podem estar associados a outras
patologias vindouras. Até o século XIX, os distúrbios mentais eram
PÁGINA: 201
juntos fazia apenas cinco anos. Aliás, segundo a autópsia, par

### Antes do LangGraph, vamos fazer o RAG "manual"

In [ ]:
!pip install -U langchain-huggingface
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [30]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [31]:
llm_hf = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    max_new_tokens=512,
    temperature=0
)

llm = ChatHuggingFace(llm=llm_hf)

In [32]:
contexto = "\n\n".join(
    documento.page_content
    for documento in documentos_encontrados
)

In [33]:
prompt = f"""
Você é um assistente que responde perguntas utilizando
exclusivamente o contexto fornecido.

Contexto:
{contexto}

Pergunta:
{pergunta}

Responda de forma clara e objetiva.
Se a informação não estiver no contexto, diga que
não encontrou a informação no documento.
"""

In [34]:
resposta = llm.invoke(prompt)

print(resposta.content)

A autora afirma que tanto a genética quanto o meio ambiente exercem influência significativa na formação do ser humano. Ela destaca que:

* **Fatores biológicos/genéticos** – embora a ciência tenha identificado alguns genes associados a transtornos como a esquizofrenia, esses genes não determinam de forma absoluta o risco de doença; a influência fisiológica (hormonal, química, sináptica) pode ser “imensamente” importante.

* **Fatores ambientais/social** – os primeiros anos de vida moldam a estrutura cerebral e a interação social é necessária para manter a inteligência na velhice, indicando que o ambiente social tem papel crucial no desenvolvimento cognitivo.

Em resumo, a autora sustenta que a formação humana resulta da combinação de influências genéticas e ambientais, sendo ambas relevantes, mas o contexto social e fisiológico pode ser tão determinante quanto os próprios genes.


### 1. Criar o estado


In [35]:
from typing_extensions import TypedDict


class EstadoRAG(TypedDict):
    pergunta: str
    contexto: str
    resposta: str

### 2. Criar o nó Retriever

In [36]:
def buscar_documentos(state: EstadoRAG):
    pergunta = state["pergunta"]

    documentos_encontrados = retriever.invoke(pergunta)

    contexto = "\n\n".join(
        documento.page_content
        for documento in documentos_encontrados
    )

    return {
        "contexto": contexto
    }

3. Criar o nó Gerador

In [37]:
def gerar_resposta(state: EstadoRAG):

    pergunta = state["pergunta"]
    contexto = state["contexto"]

    prompt = f"""
Você é um assistente que responde perguntas utilizando
exclusivamente o contexto fornecido.

Contexto:
{contexto}

Pergunta:
{pergunta}

Responda de forma clara e objetiva.

Se a informação não estiver no contexto,
diga que não encontrou a informação no documento.
"""

    resposta = llm.invoke(prompt)

    return {
        "resposta": resposta.content
    }

### 4. Criar o grafo

In [39]:
from langgraph.graph import StateGraph, START, END


workflow = StateGraph(EstadoRAG)

In [40]:
workflow.add_node(
    "retriever",
    buscar_documentos
)

workflow.add_node(
    "gerador",
    gerar_resposta
)

### 5. Conectar os nós

In [41]:
workflow.add_edge(
    START,
    "retriever"
)

In [42]:
workflow.add_edge(
    "retriever",
    "gerador"
)

In [43]:
workflow.add_edge(
    "gerador",
    END
)

In [44]:
app_rag = workflow.compile()

### 6. Executar

In [45]:
pergunta = """
O que a autora fala sobre a influência da genética
e do meio na formação do ser humano?
"""

In [46]:
estado_inicial = {
    "pergunta": pergunta,
    "contexto": "",
    "resposta": ""
}

In [49]:
resultado = app_rag.invoke(estado_inicial)
print(resultado["resposta"])

A autora afirma que a formação do ser humano resulta da interação entre fatores genéticos e ambientais, mas destaca que o meio – sobretudo as primeiras experiências de vida e a interação social – tem um peso decisivo.  

- **Genética:** embora existam genes associados a transtornos como a esquizofrenia, “nenhum desses genes te predispõe a sofrer de determinada doença tanto quanto a cor do seu passaporte”, indicando que a predisposição genética não determina sozinha o destino da pessoa.  

- **Meio / ambiente:** os primeiros anos de vida “danificam a estrutura do cérebro” se não houver estímulos adequados, e a necessidade de interação social é fundamental para manter a inteligência na velhice. Além disso, desequilíbrios fisiológicos (hormonais, químicos, sinápticos) podem gerar sintomas claros, mostrando a grande influência do ambiente biológico.

Em resumo, a autora sustenta que tanto a genética quanto o meio influenciam o desenvolvimento humano, mas a influência do meio – social, ambi

### O grafo acima é um RAG linear:
Pergunta → Busca → Geração → Fim

# Grafo RAG com decisão, loop e avaliação

### Etapa 1 — vamos criar o estado

In [51]:
class EstadoRAG(TypedDict):
    pergunta: str
    contexto: str
    resposta: str
    tentativas: int

| Campo        | Serve para                              |
| ------------ | --------------------------------------- |
| `pergunta`   | pergunta original do usuário            |
| `contexto`   | trechos encontrados no livro            |
| `resposta`   | resposta final do modelo                |
| `tentativas` | controlar quantas vezes fizemos a busca |


In [54]:
from typing_extensions import TypedDict

class EstadoRAG(TypedDict):
    pergunta: str
    contexto: str
    resposta: str
    tentativas: int
    avaliacao: str

### Etapa 2 — o avaliador

In [55]:
def avaliar_contexto(state: EstadoRAG):
    pergunta = state["pergunta"]
    contexto = state["contexto"]

    prompt = f"""
Você é um avaliador de documentos.

Analise se o contexto abaixo contém informações
relevantes para responder à pergunta.

Pergunta:
{pergunta}

Contexto:
{contexto}

Responda SOMENTE com uma destas opções:

SIM
NAO

Responda SIM se o contexto contiver informações
que possam ajudar a responder à pergunta.

Responda NAO se o contexto não for relevante
ou não tiver informações suficientes.
"""

    resposta = llm.invoke(prompt)

    avaliacao = resposta.content.strip().upper()

    return {"avaliacao": avaliacao}

### Etapa 3 — transformar a avaliação em uma decisão

In [56]:
def decidir_proximo_passo(state):
    if state["avaliacao"] == "SIM":
        return "gerador"

    return "reformular"

### Etapa 4 — nó de reformulação

In [57]:
def reformular_pergunta(state: EstadoRAG):
    pergunta_atual = state["pergunta"]

    prompt = f"""
Você é um assistente especializado em melhorar perguntas
para sistemas de busca em documentos.

A pergunta atual é:

{pergunta_atual}

Crie uma nova versão da pergunta que seja mais específica
e facilite encontrar a informação correta em um livro.

Não responda à pergunta.
Apenas reformule a pergunta.

Pergunta reformulada:
"""

    resposta = llm.invoke(prompt)

    nova_pergunta = resposta.content.strip()

    return {
        "pergunta": nova_pergunta
    }

### Etapa 5 — precisamos contar as tentativas

In [58]:
def buscar_documentos(state: EstadoRAG):
    pergunta = state["pergunta"]
    tentativas = state["tentativas"]

    documentos_encontrados = retriever.invoke(pergunta)

    contexto = "\n\n".join(
        documento.page_content
        for documento in documentos_encontrados
    )

    return {
        "contexto": contexto,
        "tentativas": tentativas + 1
    }

### Etapa 6 — vamos melhorar nossa decisão

In [59]:
def decidir_proximo_passo(state: EstadoRAG):

    if state["avaliacao"] == "SIM":
        return "gerador"

    if state["tentativas"] >= 3:
        return "gerador"

    return "reformular"

### Etapa 7 — construir o StateGraph

In [60]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(EstadoRAG)

In [61]:
workflow.add_node("retriever", buscar_documentos)
workflow.add_node("avaliador", avaliar_contexto)
workflow.add_node("reformular", reformular_pergunta)
workflow.add_node("gerador", gerar_resposta)

### Etapa 8 — ligar os nós

In [62]:
workflow.add_edge(START, "retriever")

In [63]:
workflow.add_edge("retriever", "avaliador")

In [64]:
workflow.add_conditional_edges(
    "avaliador",
    decidir_proximo_passo
)

### Etapa 9 — fechar o loop

In [65]:
workflow.add_edge("reformular", "retriever")

### Etapa 10 — finalizar o caminho do gerador

In [66]:
workflow.add_edge("gerador", END)

In [67]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(EstadoRAG)

# Nós do grafo
workflow.add_node("retriever", buscar_documentos)
workflow.add_node("avaliador", avaliar_contexto)
workflow.add_node("reformular", reformular_pergunta)
workflow.add_node("gerador", gerar_resposta)

# Caminho inicial
workflow.add_edge(START, "retriever")

# Depois de buscar, avaliar
workflow.add_edge("retriever", "avaliador")

# O avaliador decide para onde ir
workflow.add_conditional_edges(
    "avaliador",
    decidir_proximo_passo
)

# Se o contexto não for suficiente,
# reformulamos a pergunta e buscamos novamente
workflow.add_edge("reformular", "retriever")

# Quando temos uma resposta, terminamos
workflow.add_edge("gerador", END)

In [68]:
app_rag = workflow.compile()

In [69]:
pergunta = """
O que a autora fala sobre a influência da genética
e do meio na formação do ser humano?
"""

estado_inicial = {
    "pergunta": pergunta,
    "contexto": "",
    "resposta": "",
    "tentativas": 0,
    "avaliacao": ""
}

In [71]:
resultado = app_rag.invoke(estado_inicial)

print("PERGUNTA FINAL:")
print(resultado["pergunta"])

print("\nTENTATIVAS:")
print(resultado["tentativas"])

print("\nAVALIAÇÃO:")
print(resultado["avaliacao"])

print("\nRESPOSTA:")
print(resultado["resposta"])

PERGUNTA FINAL:

O que a autora fala sobre a influência da genética
e do meio na formação do ser humano?


TENTATIVAS:
1

AVALIAÇÃO:
SIM

RESPOSTA:
A autora afirma que tanto a genética quanto o meio ambiente influenciam a formação do ser humano, mas destaca que os fatores ambientais – especialmente a interação social e as condições de vida (como “a cor do seu passaporte”) – podem ter um impacto ainda maior que os genes estudados até hoje. Em resumo, ela reconhece a importância da herança genética, porém ressalta que o contexto social e fisiológico exerce uma influência imensa e, muitas vezes, mais determinante no desenvolvimento humano.


### Teste 1 — pergunta totalmente fora do livro

In [72]:
pergunta = """
Como fazer uma receita tradicional de pão de queijo?
"""

estado_inicial = {
    "pergunta": pergunta,
    "contexto": "",
    "resposta": "",
    "tentativas": 0,
    "avaliacao": ""
}

resultado = app_rag.invoke(estado_inicial)

print("PERGUNTA FINAL:")
print(resultado["pergunta"])

print("\nTENTATIVAS:")
print(resultado["tentativas"])

print("\nAVALIAÇÃO:")
print(resultado["avaliacao"])

print("\nRESPOSTA:")
print(resultado["resposta"])

PERGUNTA FINAL:
Qual é a receita tradicional mineira de pão de queijo, incluindo a lista completa de ingredientes com medidas precisas (em gramas ou colheres), o rendimento (número de porções), o tempo total de preparo e cozimento, e um passo‑a‑passo detalhado para a execução da receita?

TENTATIVAS:
3

AVALIAÇÃO:
NAO

RESPOSTA:
Não encontrei a informação no documento.


### Teste 2 — pergunta propositalmente vaga

In [73]:
pergunta = """
Quais fatores influenciam a formação do ser humano?
"""

estado_inicial = {
    "pergunta": pergunta,
    "contexto": "",
    "resposta": "",
    "tentativas": 0,
    "avaliacao": ""
}

resultado = app_rag.invoke(estado_inicial)

print("PERGUNTA FINAL:")
print(resultado["pergunta"])

print("\nTENTATIVAS:")
print(resultado["tentativas"])

print("\nAVALIAÇÃO:")
print(resultado["avaliacao"])

print("\nRESPOSTA:")
print(resultado["resposta"])

PERGUNTA FINAL:

Quais fatores influenciam a formação do ser humano?


TENTATIVAS:
1

AVALIAÇÃO:
SIM

RESPOSTA:
**Fatores que influenciam a formação do ser humano (conforme o texto):**

1. **Hereditariedade (genética)** – a carga biológica que recebemos ao nascer.  
2. **Meio/ambiente** – as condições externas que alteram a química do organismo.  
3. **Socialização** – o processo de aprendizado e interação com outros indivíduos.  
4. **Imaginação** – a capacidade de criar representações internas que moldam percepções e decisões.  
5. **Química cerebral** – influenciada tanto pelos genes quanto pelo meio, regula emoções e comportamento.  

Esses são os elementos mencionados no documento que contribuem para a formação do ser humano.
